# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print metadata information
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published on: {getattr(meta, 'datePublished', None)}")
print(f"Version: {getattr(meta, 'version', None)}\n")
print(f"Dataset License: {getattr(meta, 'license', None)}\n")
print(f"Citation: {getattr(meta, 'citeAs', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use `@id` fields, which uniquely identify entities in the Croissant schema (record sets, fields, columns, etc.).

In [ ]:
# List all record sets. Their `@id` fields are required to access data.
record_sets = dataset.metadata.recordSets
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', None)}")

# As an example, show all fields for each record set
for rs in record_sets:
    fields = rs.get('field', [])
    print(f"\nFields for record set '@id': {rs['@id']}")
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        if isinstance(f, dict):
            print(f"  - field @id: {f.get('@id')}, name: {f.get('name')}")
        else:
            print(f"  - field: {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. (Use the record set and field `@id`s discovered above.)

In [ ]:
# Here we extract all record set @id fields
record_set_ids = [rs['@id'] for rs in record_sets]
print("Extracting data from these record sets:", record_set_ids)

dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from record set '@id': {rs_id}")
        else:
            print(f"No records found for record set '@id': {rs_id}")
    except Exception as e:
        print(f"Failed to load record set '@id': {rs_id}: {e}")

# Choose the first record set for demonstration if available
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set '@id': {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    main_rs_id = None
    print("No dataframes available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. All field and column references must use `@id` strings.

In [ ]:
from pandas.api.types import is_numeric_dtype

if main_rs_id is not None:
    df = dataframes[main_rs_id]
    # Try to find a numeric field in the columns
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    print(f"Selected numeric field for analysis: {numeric_field_id}")

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a non-numeric field for grouping
        group_field = None
        for col in df.columns:
            if not is_numeric_dtype(df[col]):
                group_field = col
                break
        print(f"Group field selected: {group_field}")
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing head):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("Cannot perform EDA. No loaded DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: histogram/bar plot for a numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there's a group_field, plot mean by group
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        df_grouped = df.groupby(group_field)[numeric_field_id].mean().sort_values()
        df_grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR^2 dataset, referencing all entities using their `@id` per the Croissant schema. The analysis included basic EDA and visualizations. For deeper exploration, use the field and record set `@id`s shown to manipulate or analyze any part of the dataset.